In [1]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyulog import ULog
from pyulog.px4_events import PX4Events
from pyulog.px4 import PX4ULog

In [2]:
# return the array of combined log file line
def getLogData(baseLogDir, dateTime, iteration, testCase, model="iris"):
	logPath = os.path.join(baseLogDir, dateTime, iteration, model, testCase, "log-combined.log_plain.log")
	log = None

	if os.path.exists(logPath):
		with open(logPath, "r") as f:
			log = f.readlines()

	return log

In [3]:
# return the ulg file name parsed from the combined log
def findUlgName(log):
	pattern = r"INFO\s+\[logger\]\s+Opened full log file:\s+(.*\.ulg)"
	ulg_file_name = ""

	match = re.search(pattern, log)

	if match:
		ulg_file_name = match.group(1)

	return ulg_file_name

In [4]:
# return the ulog parsed from the ulg file
def getUlogData(baseUlgDir, ulgFileName):
	ulog = None

	normPath = os.path.normpath(ulgFileName)
	ulgPath = os.path.join(baseUlgDir, normPath)

	if os.path.exists(ulgPath):
		ulog = ULog(ulgPath)

	return ulog

In [ ]:
# ulg, combined log default location
baseLogDir = os.path.expanduser("~/PX4-Autopilot/logs")
baseUlgDir = os.path.expanduser("~/PX4-Autopilot/build/px4_sitl_default/tmp_mavsdk_tests/rootfs")

# combined log path data
experimentDateTime = "2025-03-10T17-34-40Z"
testCase = "bias_{x_0.03_y_0.00_z_0.00}_hold_20m"

# get the combined log data
combinedLog = getLogData(baseLogDir, experimentDateTime, "001", testCase)

# parse the ulg file name from the combined log
ulgFileName = findUlgName("".join(combinedLog))

# get the ulog data
ulog = getUlogData(baseUlgDir, ulgFileName)


# get the dataset from the ulog
df = ulog.get_dataset("vehicle_attitude_groundtruth")
print("pandas:", pd.DataFrame(df.data))

numpy: {'timestamp': array([ 18496000,  18500000,  18508000, ..., 145656000, 145668000,
       145676000], dtype=uint64), 'timestamp_sample': array([0, 0, 0, ..., 0, 0, 0], dtype=uint64), 'q[0]': array([7.071098e-01, 7.071098e-01, 7.071098e-01, ..., 7.788375e-06,
       7.806366e-06, 7.818200e-06], dtype=float32), 'q[1]': array([3.927511e-04, 3.928393e-04, 3.930157e-04, ..., 8.855120e-01,
       8.855120e-01, 8.855120e-01], dtype=float32), 'q[2]': array([3.9303757e-04, 3.9312578e-04, 3.9330218e-04, ..., 4.6462628e-01,
       4.6462628e-01, 4.6462628e-01], dtype=float32), 'q[3]': array([7.0710999e-01, 7.0710999e-01, 7.0710999e-01, ..., 1.0067440e-04,
       1.0074057e-04, 1.0078740e-04], dtype=float32), 'delta_q_reset[0]': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32), 'delta_q_reset[1]': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32), 'delta_q_reset[2]': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32), 'delta_q_reset[3]': array([0., 0., 0., ..., 0., 0., 0.], dtype=float3

In [ ]:
# express the dataset as numpy
print("numpy:", np.array(df.data))

# get the events and the mav type from the ulog
px4Events = PX4Events().get_logged_events(ulog)
px4ULog = PX4ULog(ulog).get_mav_type()
print(px4Events)
print(px4ULog)